In [1]:
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error

/Users/akhilendra.singh/Documents/Trip-Duration-Prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet("../data/green_tripdata_2026-02.parquet")

In [3]:
df.isnull().sum()

VendorID                     0
lpep_pickup_datetime         0
lpep_dropoff_datetime        0
store_and_fwd_flag        5387
RatecodeID                5387
PULocationID                 0
DOLocationID                 0
passenger_count           5387
trip_distance                0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
ehail_fee                37373
improvement_surcharge        0
total_amount                 0
payment_type              5387
trip_type                 5388
congestion_surcharge      5387
cbd_congestion_fee           0
dtype: int64

In [4]:
df.drop('ehail_fee',axis=1,inplace=True)

In [5]:
df = df.dropna()

In [6]:
df.shape

(31985, 20)

In [7]:
df.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'payment_type', 'trip_type', 'congestion_surcharge',
       'cbd_congestion_fee'],
      dtype='str')

In [8]:
df['duration'] = ((df['lpep_dropoff_datetime']-df['lpep_pickup_datetime']).dt.total_seconds())/60

In [9]:
df['duration']# This is the duration in minutes.

0         5.016667
1         7.483333
2        42.550000
3         9.300000
4        13.650000
           ...    
31981    17.883333
31982    14.283333
31983    15.050000
31984     8.566667
31985    15.283333
Name: duration, Length: 31985, dtype: float64

In [10]:
#Since most of thr duration is below 60 minutes , hence keep it below 60 minutes
df = df[(df['duration']>=0) & (df['duration']<=60)]

In [11]:
df = df[df['passenger_count']>0]

In [12]:
df['pickup_hour'] = df['lpep_pickup_datetime'].dt.hour

In [13]:
df['pickup_day'] = df['lpep_pickup_datetime'].dt.day

In [14]:
df['pickup_month'] = df['lpep_pickup_datetime'].dt.month

In [15]:
df['pickup_weekday'] = df['lpep_pickup_datetime'].dt.weekday

In [16]:
df['PU_DO'] = ((df['PULocationID'].astype(str))+"_"+ (df['DOLocationID'].astype(str)))

In [17]:
df.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'payment_type', 'trip_type', 'congestion_surcharge',
       'cbd_congestion_fee', 'duration', 'pickup_hour', 'pickup_day',
       'pickup_month', 'pickup_weekday', 'PU_DO'],
      dtype='str')

In [18]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,duration,pickup_hour,pickup_day,pickup_month,pickup_weekday,PU_DO
0,2,2026-02-01 00:58:25,2026-02-01 01:03:26,N,1.0,82,129,1.0,0.50,6.5,...,1.0,1.0,0.0,0.00,5.016667,0,1,2,6,82_129
1,2,2026-02-01 00:05:50,2026-02-01 00:13:19,N,1.0,33,65,1.0,1.08,9.3,...,1.0,1.0,0.0,0.00,7.483333,0,1,2,6,33_65
2,1,2026-02-01 00:45:54,2026-02-01 01:28:27,N,5.0,226,143,1.0,5.20,75.0,...,1.0,2.0,0.0,0.75,42.550000,0,1,2,6,226_143
3,2,2026-02-01 00:06:38,2026-02-01 00:15:56,N,1.0,74,235,1.0,4.70,20.5,...,2.0,1.0,0.0,0.00,9.300000,0,1,2,6,74_235
4,2,2026-02-01 00:37:31,2026-02-01 00:51:10,N,1.0,120,119,1.0,2.47,16.3,...,2.0,1.0,0.0,0.00,13.650000,0,1,2,6,120_119


In [19]:
numerical_features = [

        # time features
        "pickup_hour",
        "pickup_day",
        "pickup_month",
        "pickup_weekday",
        "passenger_count",
        "trip_distance",
    ]

categorical_features = [
    "PULocationID",
    "DOLocationID",
    "PU_DO"
]

In [20]:
X = df[numerical_features+categorical_features]
y = df['duration']

In [21]:
X.head()

,pickup_hour,pickup_day,pickup_month,pickup_weekday,passenger_count,trip_distance,PULocationID,DOLocationID,PU_DO
0,0,1,2,6,1.0,0.50,82,129,82_129
1,0,1,2,6,1.0,1.08,33,65,33_65
2,0,1,2,6,1.0,5.20,226,143,226_143
3,0,1,2,6,1.0,4.70,74,235,74_235
4,0,1,2,6,1.0,2.47,120,119,120_119


In [23]:
test_data_2,test_data_3,test_data_2_duration,test_data_3_duration = train_test_split(X,y,test_size=0.5,random_state=42)


In [24]:
test_data_2.to_csv("./test_data_2.csv")

In [25]:
test_data_3.to_csv("./test_data_3.csv")

In [28]:
test_data_2_duration.to_csv("./test_data_2_duration.csv")

In [29]:
test_data_3_duration.to_csv("./test_data_3_duration.csv")